In [1]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import UnstructuredPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores.utils import filter_complex_metadata
from dotenv import load_dotenv
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

d:\programming\RAG implimentation\Mizhavu Chatbot\service\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Local\Temp\ipykernel_28276\4084053857.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredPDFLoader, DirectoryLoader


In [2]:
load_dotenv()

True

In [3]:
current_dir=os.getcwd()
file_path=os.path.join(current_dir,"Documents")
persistant_dir=os.path.join(current_dir,"database","Chroma_db")

In [4]:
loader = DirectoryLoader(
    path=file_path,
    glob="**/*.pdf", 
    loader_cls=UnstructuredPDFLoader,
    loader_kwargs={'mode': 'elements', 'strategy': 'fast'}
)
docs = loader.load()

No languages specified, defaulting to English.
No languages specified, defaulting to English.
No languages specified, defaulting to English.
No languages specified, defaulting to English.
No languages specified, defaulting to English.
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 85 0 (offset 0)
Ignoring wrong pointing object 87 0 (offset 0)
Ignoring wrong pointing object 89 0 (offset 0)
Ignoring wrong pointing object 208 0 (offset 0)
Ignoring wrong pointing object 227 0 (offset 0)
Ignoring wrong pointing object 229 0 (offset 0)
No languages specified, defaulting to English.
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 53 0 (offset 0)
Ignoring wrong pointing object 84 0 (offset 0)
Ignoring wrong pointing object 92 0 (offset 0)
Ignoring wrong pointing object 108 0 (offset 0)
No languages specified, defaulting to English.
No langua

In [5]:
docs

[Document(metadata={'source': 'd:\\programming\\RAG implimentation\\Mizhavu Chatbot\\service\\Documents\\21043748053.DivyaNedungadi.pdf', 'coordinates': {'points': ((71.904, 37.30463999999995), (71.904, 48.34463999999991), (233.45, 48.34463999999991), (233.45, 37.30463999999995)), 'system': 'PixelSpace', 'layout_width': 595.32, 'layout_height': 841.92}, 'file_directory': 'd:\\programming\\RAG implimentation\\Mizhavu Chatbot\\service\\Documents', 'filename': '21043748053.DivyaNedungadi.pdf', 'last_modified': '2025-11-04T01:21:27', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/pdf', 'category': 'Header', 'element_id': 'e5f2eec0dab8edd103e7a5d35162ff03'}, page_content='Vol. 8, Issue -2 (July 2019) pp. 6-10'),
 Document(metadata={'source': 'd:\\programming\\RAG implimentation\\Mizhavu Chatbot\\service\\Documents\\21043748053.DivyaNedungadi.pdf', 'coordinates': {'points': ((446.14, 37.30463999999995), (446.14, 62.92247999999995), (528.4200000000001, 62.92247999999995), (5

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200
)

In [7]:
chunks = text_splitter.split_documents(docs)
chunks = filter_complex_metadata(chunks)

In [8]:
chunks

[Document(metadata={'source': 'd:\\programming\\RAG implimentation\\Mizhavu Chatbot\\service\\Documents\\21043748053.DivyaNedungadi.pdf', 'file_directory': 'd:\\programming\\RAG implimentation\\Mizhavu Chatbot\\service\\Documents', 'filename': '21043748053.DivyaNedungadi.pdf', 'last_modified': '2025-11-04T01:21:27', 'page_number': 1, 'filetype': 'application/pdf', 'category': 'Header', 'element_id': 'e5f2eec0dab8edd103e7a5d35162ff03'}, page_content='Vol. 8, Issue -2 (July 2019) pp. 6-10'),
 Document(metadata={'source': 'd:\\programming\\RAG implimentation\\Mizhavu Chatbot\\service\\Documents\\21043748053.DivyaNedungadi.pdf', 'file_directory': 'd:\\programming\\RAG implimentation\\Mizhavu Chatbot\\service\\Documents', 'filename': '21043748053.DivyaNedungadi.pdf', 'last_modified': '2025-11-04T01:21:27', 'page_number': 1, 'filetype': 'application/pdf', 'category': 'Header', 'element_id': '5233ff91b6e50ad1f7265b27f3e74c76'}, page_content='Sangeet Galaxy ISSN: 2319-9695'),
 Document(metadat

In [9]:
embedding=GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
    )
    

In [10]:
index_name = os.getenv("PINECONE_INDEX_NAME")
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embedding,
    index_name=index_name
)